# Road surface risk after rain

For each road in the park: how likely is it to be muddy or hard to pass, given the rain of the last days?

1. **Roads:** OpenStreetMap through OSMnx, without footways and paths.
2. **Rain:** daily rainfall per satellite pixel from Earth Engine. Use CHIRPS for past dates (it is published a few weeks late) or IMERG for recent days.
3. **Rain index:** recent days count more. Each day's rain is multiplied by 0.84^days_ago (antecedent precipitation index), so rain from a week ago has mostly dried out.
4. **Risk:** `surface_risk = min(1, base_road_risk + road_vulnerability × rain_risk)`. A track reacts fully to rain, a primary road barely does. A road crossing several pixels takes the wettest one.

In [1]:
import ee

ee.Authenticate()
ee.Initialize(project='project-0cf410a3-f35f-4911-9c5')

PLACE_NAME = "Tarangire National Park, Tanzania"

C:\Users\Juan\scoop\apps\python313\current\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


In [ ]:
from datetime import date

import ipywidgets as widgets
from IPython.display import clear_output, display

from safari_explorer.rainfall import CHIRPSProvider, IMERGProvider
from safari_explorer.risk_map import create_risk_map

providers = {"CHIRPS (past dates)": CHIRPSProvider(), "IMERG (recent days)": IMERGProvider()}

provider_picker = widgets.Dropdown(options=list(providers), description="Rain data:")
date_picker = widgets.DatePicker(description="Date:", value=date(2024, 4, 30))
lookback_days_input = widgets.IntText(value=5, description="Lookback days:")
load_button = widgets.Button(description="Load Map", button_style="primary")
output = widgets.Output()


def on_render_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            display(create_risk_map(PLACE_NAME, date_picker.value, lookback_days_input.value,
                                    providers[provider_picker.value]))
        except ValueError as e:
            print(f"rain data error: {e}")


load_button.on_click(on_render_clicked)
display(widgets.HBox([provider_picker, date_picker, lookback_days_input, load_button]), output)

Output()